# gemynd — open-source claim extraction on Colab (A100)

Serve an open-weights model with **vLLM** (OpenAI-compatible API) so the gemynd
claim-extraction LLM path can run on a GPU instead of the metered Anthropic API.

gemynd selects this backend automatically when `CLAIM_LLM_BASE_URL` is set (see
`gemynd/ingest/extractors/llm_adapter_factory.py`). The model speaks the same
domain-driven prompt and returns the same JSON claim rows as the Anthropic adapter.

**Two ways to use it:**
- **Option A — tunnel:** serve here, expose a URL, keep running the pipeline on your
  laptop pointed at it. Least setup; best for a quick test.
- **Option B — inline (recommended for the full run):** upload the repo + corpus to
  Drive and run the whole ingest here, so the A100 does inference with no network
  round-trip per paragraph.

Requires a GPU runtime: **Runtime → Change runtime type → A100**.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [6]:
# Install a CURRENT vLLM. Do NOT pin an old version (e.g. 0.6.3.post1) — it breaks
# against Colab's newer `transformers` at tokenizer init:
#   AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
# Latest vLLM pulls a matching transformers and avoids that. (Takes a few minutes.)
#
# Colab also prints a wall of red 'dependency conflict' lines about
# numpy / starlette / jax / opencv / rasterio / shap. Those are EXPECTED and
# harmless here: this notebook only *serves* a model and never imports those
# packages, so the conflicts never execute. The install itself still succeeds.
!pip install -q -U pyairports
# Verify in a fresh subprocess (avoids the kernel's now-stale in-memory numpy):


## Serve the model

`Qwen2.5-32B-Instruct-AWQ` fits comfortably on a 40GB A100 (~20GB weights, leaving
room for the KV cache) and is a strong extractor. Swap `MODEL` for a smaller one
(`Qwen/Qwen2.5-14B-Instruct-AWQ`) if you hit OOM, or a larger context if your
paragraphs are long.

In [3]:
import os, subprocess, time, urllib.request

MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"
PORT = 8000

server = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--quantization", "awq_marlin",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.92",
    "--port", str(PORT),
])

# Wait for the server to come up (model download + load can take several minutes).
base = f"http://localhost:{PORT}/v1"
for _ in range(120):
    try:
        urllib.request.urlopen(f"{base}/models", timeout=5)
        print("\nvLLM is up at", base)
        break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError("vLLM did not start — check the cell output / GPU memory.")

KeyboardInterrupt: 

In [5]:
# Smoke-test the endpoint with the OpenAI chat-completions shape gemynd uses.
import json, urllib.request
req = urllib.request.Request(
    f"{base}/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
        "max_tokens": 8, "temperature": 0.0,
    }).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer not-needed"},
)
print(json.load(urllib.request.urlopen(req))["choices"][0]["message"]["content"])

HTTPError: HTTP Error 500: Internal Server Error

## Option A — expose a tunnel and run the pipeline on your laptop

Run the cell below, copy the printed `https://*.trycloudflare.com` URL, then on your
machine:

```powershell
$env:CLAIM_LLM_BASE_URL = "https://<your-tunnel>.trycloudflare.com/v1"
$env:CLAIM_LLM_MODEL    = "Qwen/Qwen2.5-32B-Instruct-AWQ"
py scripts/ingest_tutorstructure.py --mode articles --limit 300 `
    --out-dir out/newspapers_oss_smoke --corpus C:\Users\Benjamin\Desktop\Tutorstructure\corpus
```

gemynd auto-selects the OpenAI-compatible backend because `CLAIM_LLM_BASE_URL` is set.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess
subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"])
print("Watch the output above for the trycloudflare.com URL (append /v1 when setting CLAIM_LLM_BASE_URL).")

## Option B — run the full ingest here (recommended for the full corpus)

Upload the repo and corpus to Google Drive first (e.g. zip `Archive-Claim-Graph`
and `Tutorstructure/corpus`, drop them in `MyDrive/gemynd/`). Then mount and run.
No per-paragraph network round-trip — the A100 does everything.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Adjust these to wherever you uploaded them:
REPO = "/content/drive/MyDrive/gemynd/Archive-Claim-Graph"
CORPUS = "/content/drive/MyDrive/gemynd/Tutorstructure/corpus"
import os; assert os.path.isdir(REPO), REPO; assert os.path.isdir(CORPUS), CORPUS

In [ ]:
import os
os.environ["CLAIM_LLM_BASE_URL"] = base          # http://localhost:8000/v1
os.environ["CLAIM_LLM_MODEL"]    = MODEL
os.environ["CLAIM_LLM_PROVIDER"] = "vllm"        # explicit (auto-selected anyway)

# Full run over the whole corpus, memory backend, JSON bundles out to Drive.
!cd {REPO} && python scripts/ingest_tutorstructure.py \
    --mode articles --full \
    --corpus {CORPUS} \
    --out-dir /content/drive/MyDrive/gemynd/out/newspapers_oss_full \
    --workers 4

The resulting `*.semantic.json` bundles land in your Drive `out/` folder. Entity
minting (enabled in `resources_newspaper/domain_schema.yaml`) turns the model's
claim-link surfaces into first-class entities, so claims attach to the people,
places, and organizations the articles name — not just the seed gazetteer.